# Тестовое задание в группу вычислительной химии

##### Выполнил Михаил Лоев

#### Импорты

In [10]:
from pathlib import Path

import rdkit
import numpy as np
from openff.toolkit import  Molecule, ForceField, Quantity
from openff.interchange import Interchange


#### Входные данные

В `sdf_file` следует передать путь к SDF файлу  

В `output_dir` следует передать путь папке, в которую будут сложены все параметризованные молекулы 

В `forcefield` нужно передать в качестве аргумента `ForceField()` одно из доступных в [OpenFF](https://openforcefield.org/force-fields/force-fields/) силовых полей

In [11]:
sdf_file = "./example.sdf"
output_dir = "./output/"
forcefield = ForceField("openff-2.3.0.offxml")

#### Функции

В целом последовательность шагов программы выглядит следующим образом:   

1. С помощью `rdkit.Chem.SDMolSupplier()` парсится SDF файл со структурами   


2. Для каждой структуры в цикле выполняются функции, описанные в ячейке ниже  


3. `obtain_name(rdkit_mol, mol_counter)` получает из молекулы (`rdkit_mol`) ее имя. Сначала проверяется пользовательское поле "Molecule Name", потом, если там не оказывается названия, проверяется название в шапке через поле "_Name". Если по какой-то причине и в нем ничего не оказывается (либо поле отсутсвует, либо пустое), молекуле присваивается название вида mol<номер молекулы в SDF файле> (`mol_counter`)

4. `make_dir(dir_path, mol_counter)` с использованием полученного имени создает папку в указанной для вывода директории (`dir_path` содержит путь до директории с аутпутом, соединенный с названием молекулы). Если в SDF файле встречаются еще молекулы с одним и тем же названием, создается папка с названием вида <имя молекулы>_<номер молекулы в SDF файле> (`mol_counter`). Это менее изящно, чем если бы создавалась папка <имя молекулы>_2/3/4/..., но не требует отслеживания дупликатов для каждой молекулы отдельно

5. Дальше в `create_interchange(rdkit_mol, forcefield)` молекула RDKit (`rdkit_mol`) переводится в объект Molecule openff-toolkit. На его основе вместе с силовым полем `forcefield` создается топология и формируется объект Interchange. Функция возвращает объект Interchange. 

6. `check_rmsd(rdkit_mol, inter)` проводит проверку совпадения координат в полученном Interchange (`inter`) объекте с координатами в молекуле RDKit (`rdkit_mol`). Потом на основе этих данных будет посчитан суммарный RMSD по всем молекулам, содержащимся в SDF файле, чтобы продемонстрировать, что координаты были полностью сохранены

7. Наконец, `create_top_gro(molname, interchange, outpath, decim)` создает в полученной в п. 4 папке (`outpath`) два файла: с топологией и координатами (я делал .gro), называющиеся в соответсвие с названием молекулы, полученным в п. 1 (`molname`). Для сохранения точности исходных координат при переходе из ангстрем в нанометры характерные для .gro файлов я задал параметр `decimal=5` (можно настроить через `decim`) Предварительно создается кубическая ячейка. ([зачем](https://docs.openforcefield.org/projects/interchange/en/stable/using/edges.html#some-mm-engines-do-not-support-no-cutoff-non-bonded-methods))

8. Все эти функции выполняются в теле основной функции `sdf_to_gromacs(rdkit_mol, mol_counter, forcefield, decim = 5, outdir = ".")`

In [12]:
def obtain_name (rdkit_mol, mol_counter):
    
    try:
      mol_name = rdkit_mol.GetProp("Molecule Name")
    except KeyError: 
      if rdkit_mol.HasProp("_Name") and rdkit_mol.GetProp("_Name").strip() != "":
        mol_name = rdkit_mol.GetProp("_Name")
      else: 
        mol_name = f"mol{mol_counter}"

    return mol_name

def make_dir (dir_path, mol_counter):

    try: 
        Path.mkdir(dir_path)
    except FileExistsError: 
        dir_path =dir_path.with_stem(f"{dir_path.stem}_{mol_counter}")
        Path.mkdir(dir_path) 

    return None 


def create_interchange (rdkit_mol, forcefield):

    molecule = Molecule.from_rdkit(rdkit_mol)
    topology = molecule.to_topology()
    inter = Interchange.from_smirnoff(force_field=forcefield,topology= topology)

    return inter


def create_top_gro (molname, interchange, outpath, decim):

    interchange.box = Quantity([10, 10, 10], "nanometer")
    interchange.to_top(outpath / f"{molname}.top")
    interchange.to_gro(outpath/ f"{molname}.gro", decimal = decim)

    return None


def check_coords (rdkit_mol, inter):
    rdkit_coords = rdkit_mol.GetConformer().GetPositions()

    # OpenFF координаты → переводим в Å
    off_coords = inter.positions.m_as("angstrom")

    # RMSD
    diff = rdkit_coords - off_coords
    rmsd = np.sqrt((diff ** 2).sum(axis=1).mean())
    
    return rmsd


def sdf_to_gromacs (rdkit_mol, mol_counter, forcefield, decim = 5, outdir = "."):

    mol_name = obtain_name(rdkit_mol, mol_counter)

    out_path = Path(outdir) / mol_name
    make_dir(out_path, mol_counter)

    inter = create_interchange(rdkit_mol, forcefield)
    
    rmsd = check_coords(rdkit_mol, inter)

    create_top_gro(mol_name, inter, out_path, decim)

    
    return rmsd


#### Обработка SDF файла

Первым делом создается (если нужно) папка, в которой будут храниться все результаты

Орабатываю в контекстном менеджере для безопасности. В `rdkit.Chem.SDMolSupplier()` прописываю (помимо пути к SDF файлу `sdf_file`) `removeHs=False`, чтобы сохранить атомы водорода из SDF файла.

`mol_counter` - счетчик молекул в SDF файле

`errors` - счетчик не обработанных по той или иной причине молекул

`rmsd` - для подсчета суммарного RMSD

In [13]:
Path(output_dir).mkdir(parents=True, exist_ok=True)

with rdkit.Chem.SDMolSupplier(sdf_file, removeHs = False) as suppl:
  mol_counter = 0
  errors = 0
  rmsd = 0

  for mol in suppl:
    mol_counter+=1
    if mol is None:
      errors+=1
      continue
    
    rmsd += sdf_to_gromacs(mol, mol_counter, forcefield, outdir=output_dir)

  print(f"Processed {mol_counter} molecules")
  print(f"Failed to process {errors} molecules")
  print(f"Total RMSD of coordinates is {rmsd} Å")

[02:13:22] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
/home/domain/mlmisha/miniconda3/envs/openff-toolkit/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
[02:13:31] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
/home/domain/mlmisha/miniconda3/envs/openff-toolkit/lib/python3.13/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):
[02:13:39] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero.

Processed 54 molecules
Failed to process 0 molecules
Total RMSD of coordinates is 3.093557549836968e-14 Å
